In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/pradippokhrel45/optimizer-information/optuna1 (1).db
/kaggle/input/datasets/pradippokhrel45/coding-problem-of-alpaca-and-flytech-only/refined_train.jsonl
/kaggle/input/datasets/pradippokhrel45/llama-2-fine-tuning-datasets-for-python-code/train.jsonl


In [2]:
import shutil

shutil.copy(
    "/kaggle/input/datasets/pradippokhrel45/optimizer-information/optuna1 (1).db",
    "/kaggle/working/optuna1.db"
)

'/kaggle/working/optuna1.db'

In [3]:
!pip install trl bitsandbytes optuna accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 10.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.6 MB/s eta 0:00:00:00:0100:01


In [4]:
!pip install -q --upgrade transformers datasets peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 82.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 38.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requires transformers<5.0.0,>=4.41.0, but you have transformers 5.2.0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [ ]:
import huggingface_hub
import os
hf_token = os.getenv("hf_token")

huggingface_hub.login(token = hf_token)

In [ ]:
import os
import gc
import json
import torch
import optuna
import random
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


# ================= CONFIG =================
BASE_MODEL = "meta-llama/Llama-2-7b-chat-hf"
DATASET_FILE = "/kaggle/input/datasets/pradippokhrel45/coding-problem-of-alpaca-and-flytech-only/refined_train.jsonl"
OUTPUT_DIR = "./optuna_results"

MAX_LENGTH = 512
SEED = 42
N_TRIALS = 45

TRAIN_SUBSET_RATIO = 0.05
VAL_RATIO = 0.2


# ================= SEED =================
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()


# ================= LOAD DATA =================
print("[DEBUG] Loading dataset...")

full_dataset = load_dataset(
    "json",
    data_files=DATASET_FILE,
    split="train"
)

print("[DEBUG] Dataset size:", len(full_dataset))


subset_size = int(len(full_dataset) * TRAIN_SUBSET_RATIO)

subset_dataset = (
    full_dataset
    .shuffle(seed=SEED)
    .select(range(subset_size))
)

print("[DEBUG] Subset size:", len(subset_dataset))


val_size = int(len(subset_dataset) * VAL_RATIO)

train_size = len(subset_dataset) - val_size

train_ds = subset_dataset.select(range(train_size))
val_ds = subset_dataset.select(range(train_size, train_size + val_size))

print("[DEBUG] Train size:", len(train_ds))
print("[DEBUG] Val size:", len(val_ds))


# ================= TOKENIZER =================
print("[DEBUG] Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


# ================= TOKENIZATION (SFT MASKING) =================
def tokenize_fn(batch):

    input_ids_list = []
    labels_list = []
    attention_mask_list = []

    for prompt, code in zip(batch["prompt"], batch["code"]):

        prompt = str(prompt)
        code = str(code)

        # Full formatted instruction
        full_text = prompt + "\n" + code

        tokenized = tokenizer(
            full_text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]

        # Tokenize only prompt to mask it
        prompt_ids = tokenizer(
            prompt,
            truncation=True,
            max_length=MAX_LENGTH
        )["input_ids"]

        prompt_len = len(prompt_ids)

        labels = [-100] * len(input_ids)

        # Enable loss only on code tokens
        for i in range(prompt_len, len(input_ids)):
            if input_ids[i] != tokenizer.pad_token_id:
                labels[i] = input_ids[i]

        input_ids_list.append(input_ids)
        labels_list.append(labels)
        attention_mask_list.append(attention_mask)

    return {
        "input_ids": input_ids_list,
        "labels": labels_list,
        "attention_mask": attention_mask_list
    }


print("[DEBUG] Tokenizing train dataset...")
train_ds = train_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=train_ds.column_names
)

print("[DEBUG] Tokenizing val dataset...")
val_ds = val_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=val_ds.column_names
)


# ================= QUANT CONFIG =================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)


# ================= OBJECTIVE =================
def objective(trial):

    print(f"\n========== STARTING TRIAL {trial.number} ==========")

    lr = trial.suggest_float("learning_rate", 8e-6, 3e-5, log=True)

    scheduler = trial.suggest_categorical(
        "lr_scheduler",
        ["linear", "cosine", "cosine_with_restarts"]
    )

    lora_r = trial.suggest_categorical("lora_r", [8, 16, 32])

    lora_alpha = trial.suggest_categorical("lora_alpha", [8, 16, 32])

    lora_dropout = trial.suggest_float("lora_dropout", 0.05, 0.15)

    batch_size = trial.suggest_categorical("batch_size", [4, 8,16])

    warmup_ratio = trial.suggest_float("warmup_ratio", 0.03, 0.15)

    print("[DEBUG] Hyperparameters:", {
        "lr": lr,
        "scheduler": scheduler,
        "r": lora_r,
        "alpha": lora_alpha,
        "dropout": lora_dropout,
        "batch": batch_size,
        "warmup": warmup_ratio
    })


    gc.collect()
    torch.cuda.empty_cache()


    try:

        # ========== Load Model ==========
        print("[DEBUG] Loading model...")

        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=bnb_config,
            device_map="auto"
        )

        model = prepare_model_for_kbit_training(model)

        print("[DEBUG] Model loaded")


        # ========== LoRA ==========
        print("[DEBUG] Applying LoRA...")

        lora_config = LoraConfig(
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=[
                "q_proj",
                "k_proj",
                "v_proj",
                "o_proj",
                "gate_proj",
                "up_proj",
                "down_proj"
            ]
        )

        model = get_peft_model(model, lora_config)

        model.print_trainable_parameters()

        total_steps = int(
            len(train_ds)
            / (batch_size * 4)
            * 2
        )

        warmup_steps = int(total_steps * warmup_ratio)

        print("[DEBUG] Total steps:", total_steps)
        print("[DEBUG] Warmup steps:", warmup_steps)


        # ========== Training Args ==========
        print("[DEBUG] Building TrainingArguments...")

        args = TrainingArguments(

            output_dir=f"{OUTPUT_DIR}/trial_{trial.number}",

            num_train_epochs=2,

            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,

            gradient_accumulation_steps=4,

            learning_rate=lr,

            lr_scheduler_type=scheduler,

            warmup_steps=warmup_steps,

            logging_steps=50,

            save_strategy="no",

            bf16=False,
            fp16=True,

            weight_decay=0.01,

            max_grad_norm=1.0,

            report_to="none",

            seed=SEED,

            remove_unused_columns=False,
        )


        # ========== Trainer ==========
        print("[DEBUG] Initializing Trainer...")

        trainer = Trainer(

            model=model,

            args=args,

            train_dataset=train_ds,

            eval_dataset=val_ds,
        )


        # ========== Train ==========
        print("[DEBUG] Starting training...")

        train_result = trainer.train()

        print("[DEBUG] Training finished")

        loss = train_result.training_loss

        print("[DEBUG] Training loss:", loss)

        if loss is None or np.isnan(loss) or np.isinf(loss):
            print("[ERROR] Invalid training loss")
            return float("inf")


        # ========== Eval ==========
        print("[DEBUG] Starting evaluation...")

        metrics = trainer.evaluate()

        print("[DEBUG] Eval metrics:", metrics)

        val_loss = metrics.get("eval_loss", None)

        print("[DEBUG] Validation loss:", val_loss)

        if val_loss is None or np.isnan(val_loss) or np.isinf(val_loss):
            print("[ERROR] Invalid validation loss")
            return float("inf")


    except Exception as e:

        print(f"[FATAL ERROR] Trial {trial.number} crashed:", str(e))

        if "model" in locals():
            del model

        if "trainer" in locals():
            del trainer

        gc.collect()
        torch.cuda.empty_cache()

        return float("inf")


    del model
    del trainer

    gc.collect()
    torch.cuda.empty_cache()

    print(f"[DEBUG] Trial {trial.number} finished successfully")

    return val_loss



# ================= MAIN =================
def main():

    sampler = optuna.samplers.TPESampler(
        seed=SEED,
        multivariate=True
    )

    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=1
    )


    study = optuna.create_study(

        direction="minimize",

        sampler=sampler,

        pruner=pruner,

        study_name="llama2_qlora_safe",

        storage="sqlite:///optuna1.db",

        load_if_exists=True
    )


    print("[DEBUG] Starting Optuna optimization...")

    study.optimize(objective, n_trials=N_TRIALS)


    best = study.best_trial


    print("\n========== BEST TRIAL ==========")

    print("Value:", best.value)

    print("\nParams:")

    for k, v in best.params.items():
        print(f"{k}: {v}")


    with open("best_params.json", "w") as f:
        json.dump(best.params, f, indent=4)


# ================= RUN =================
if __name__ == "__main__":

    main()


[DEBUG] Loading dataset...


Generating train split: 0 examples [00:00, ? examples/s]

[DEBUG] Dataset size: 12522
[DEBUG] Subset size: 626
[DEBUG] Train size: 501
[DEBUG] Val size: 125
[DEBUG] Loading tokenizer...


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

...

[DEBUG] Eval metrics: {'eval_loss': 0.585334300994873, 'eval_runtime': 82.2733, 'eval_samples_per_second': 1.519, 'eval_steps_per_second': 0.389, 'epoch': 2.0}
[DEBUG] Validation loss: 0.585334300994873
[DEBUG] Trial 45 finished successfully

========== STARTING TRIAL 46 ==========
[DEBUG] Hyperparameters: {'lr': 2.5993835219092706e-05, 'scheduler': 'cosine', 'r': 8, 'alpha': 32, 'dropout': 0.09701785089929542, 'batch': 4, 'warmup': 0.12908368137757054}
[DEBUG] Loading model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[DEBUG] Model loaded
[DEBUG] Applying LoRA...
trainable params: 19,988,480 || all params: 6,758,404,096 || trainable%: 0.2958
[DEBUG] Total steps: 62
[DEBUG] Warmup steps: 8
[DEBUG] Building TrainingArguments...
[DEBUG] Initializing Trainer...
[DEBUG] Starting training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[W 2026-02-24 15:09:56,098] Trial 46 failed with parameters: {'learning_rate': 2.5993835219092706e-05, 'lr_scheduler': 'cosine', 'lora_r': 8, 'lora_alpha': 32, 'lora_dropout': 0.09701785089929542, 'batch_size': 4, 'warmup_ratio': 0.12908368137757054} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykern

KeyboardInterrupt: 